This script is written by Wouter Ipermans (Master student 2022-2023) in Matlab and translated to Python.

In [ ]:
import PyPDF2
import pandas as pd
import re
import csv
import os

In [ ]:
def get_file_name(file_path):
    """
    This function takes a file path as input and returns the file name without the extension.
    """
    file_name = os.path.basename(file_path)  # get the file name from the file path
    file_name_without_extension, extension = os.path.splitext(file_name)  # split the file name and extension
    return file_name_without_extension

In [ ]:
def get_pdf_files(path):
    """
    This function takes a path to a folder as input and returns a list of all pdf files in that folder.
    """
    pdf_files = []
    for file_name in os.listdir(path):
        if file_name.endswith('.pdf'):
            pdf_files.append(os.path.join(path, file_name))
    return pdf_files

In [ ]:
def read_string_text(lines, header):
    # Initialize an empty list to store the dataframes
    dfs = []
    # Initialize a counter to keep track of the number of dataframes read
    count = 0
   
    # Initialize an empty list to store the rows of each dataframe
    df_rows = []
    # Iterate over the lines
    found_table = False
    for line in lines:
        # Check if the line is one of the keywords that separates the dataframes
        if any([True for word in line.split() if word in header]):
            found_table = True
            df = pd.DataFrame(df_rows)
            # Append the new dataframe to the list of dataframes
            dfs.append(df)
            # Increment the counter
            count += 1
            # Clear the list of rows for the next dataframe
            df_rows = []
        else:
            # If the line is not a keyword, split it into columns and append the row to the list
            row = line.strip().split()
            if row[0] not in compounds:
                row.insert(0, 'N.F')
            df_rows.append(row)
    # Create the final dataframe with the remaining rows
    df = pd.DataFrame(df_rows)
    dfs.append(df)

    # Return the list of dataframes
    if found_table:
        return dfs
   
    return []

In [ ]:
compounds = [
"CO2",
"CH4",
"N2",
"CO",
"H2",
"O2",
"C2H6",
"C2H4",
"C2H2",
"C3H8",
"CH3OCH3",
"CH3OH",
"CH3CH2OH"]
print('compounds that can be detected:')
print('\n'.join(map(str, compounds)))

In [ ]:
path_folder = 'H:/data/co2-splitting/uhasselt/SiO2+TMAH+2-PrOH-220-09H/chromatograms/02.5s-01-plasma.rslt/'
print('folder defined is: ' + path_folder)

pdf_files = get_pdf_files(path_folder)

#loop over all pdf files found in the directory
for pdf_file in pdf_files:
   
    pdf_reader = PyPDF2.PdfReader(pdf_file)

    #extract text in a list containing each page. Each entry is one large string.
    pages_text = []
    for page in pdf_reader.pages:
        pages_text.append(page.extract_text())

    counter = 0
    #process each page string
    for page_string in pages_text:
        lines = page_string.split('\n')
       
        header = ['Compound', 'RT', 'Exp. RT', 'Area (a.u.)', 'Area (%)']
        #get list of dataframse with the tables
        dfs = read_string_text(lines, header)
        if len(dfs) != 0:
            for df in dfs:
                counter += 1
                #TODO we should add a check and see whether the df is really a table
                df.to_csv(path_folder + get_file_name(pdf_file) + '_' + str(counter) + '.csv', sep = ';')
       
        #save as csv only dfs that have the tables in it.

In [ ]:
import os
import pandas as pd
import numpy as np

# Specify the folder where the files live.
my_folder = 'C:\\Users\\woute\\Desktop\\MA-20007-500\\GC\\recal\\2023-03-16-MA-20007-500-01-03-Plasma.rslt'
outputfile = 'C:\\Users\\woute\\Desktop\\MA-20007-500\\GC\\recal\\Plasma-03_outlier.xlsx'

# Check to make sure that folder actually exists. Warn user if it doesn't.
if not os.path.isdir(my_folder):
    print(f'Error: The following folder does not exist:\n{my_folder}\nPlease specify a new folder.')
    my_folder = input('Enter new folder path: ')

# Initialize empty lists for the data
N2, CH4, CO2, CO, H2, O2, C2H6, C2H4, C2H2, C3H8, CH3OCH3, CH3OH, CH3CH2OH = ([] for _ in range(13))

# Read and process all csv files in the directory
for filename in os.listdir(my_folder):
    if filename.endswith('.csv') and 'MCR_1' not in filename:
        df = pd.read_csv(os.path.join(my_folder, filename))
        for index, row in df.iterrows():
            compound = row[2]
            if 'MCR_2' in filename:
                if compound == 'N2':
                    N2.append(row[7])
                elif compound == 'H2':
                    H2.append(row[7])
                elif compound == 'CO':
                    CO.append(row[7])
                elif compound == 'O2':
                    O2.append(row[7])
            elif 'MCR_3' in filename and index < 8:
                if compound == 'CO2':
                    CO2.append(row[7])
                elif compound == 'CH4':
                    CH4.append(row[7])
            elif 'MCR_4' in filename:
                if compound == 'C2H6':
                    C2H6.append(row[7])
                elif compound == 'C2H4':
                    C2H4.append(row[7])
                elif compound == 'C2H2':
                    C2H2.append(row[7])
            elif 'MCR_6' in filename:
                if compound == 'C3H8':
                    C3H8.append(row[7])
                elif compound == 'CH3OCH3':
                    CH3OCH3.append(row[7])
                elif compound == 'CH3OH':
                    CH3OH.append(row[7])
                elif compound == 'CH3CH2OH':
                    CH3CH2OH.append(row[7])

# Create a DataFrame for all concentrations
df_all = pd.DataFrame({
    'N2': N2,
    'CH4': CH4,
    'CO2': CO2,
    'CO': CO,
    'H2': H2,
    'O2': O2,
    'C2H6': C2H6,
    'C2H4': C2H4,
    'C2H2': C2H2,
    'C3H8': C3H8,
    'CH3OCH3': CH3OCH3,
    'CH3OH': CH3OH,
    'CH3CH2OH': CH3CH2OH
})

# Replace outliers with 0
for column in df_all.columns:
    Q1 = df_all[column].quantile(0.25)
    Q3 = df_all[column].quantile(0.75)
    IQR = Q3 - Q1
    upper_threshold = Q3 + 1.5 * IQR
    lower_threshold = Q1 - 1.5 * IQR
    df_all.loc[(df_all[column] < lower_threshold) | (df_all[column] > upper_threshold), column] = 0

# Shift the DataFrame
df_shifted = df_all.shift(9)

# Write to Excel
df_shifted.to_excel(outputfile, index=False)

print(df_shifted)
print('done')

In [ ]:
clc
clear

% Specify the folder where the files live.
myFolder = 'C:\Users\woute\Desktop\MA-20007-500\GC\recal\2023-03-16-MA-20007-500-01-03-Plasma.rslt';
outputfile = 'C:\Users\woute\Desktop\MA-20007-500\GC\recal\Plasma-03_outlier.xlsx';
% Check to make sure that folder actually exists.  Warn user if it doesn't.
if ~isfolder(myFolder)
    errorMessage = sprintf('Error: The following folder does not exist:\n%s\nPlease specify a new folder.', myFolder);
    uiwait(warndlg(errorMessage));
    myFolder = uigetdir(); % Ask for a new one.
    if myFolder == 0
         % User clicked Cancel
         return;
    end
end

% Get a list of all files in the folder with the desired file name pattern.
filePattern = fullfile(myFolder, '*.csv'); % Change to whatever pattern you need.
theFiles = dir(filePattern);
files_table = struct2table(theFiles); % convert the struct array to a table
sortedT = sortrows(files_table);
N2 = [];
CH4 = [];
CO2 = [];
CO = [];
H2 = [];
O2 = [];
C2H6 = [];
C2H4 = [];
C2H2 = [];
C3H8 = [];
CH3OCH3 = [];
CH3OH = [];
CH3CH2OH = [];

for k = 1 : length(theFiles)
    baseFileName = theFiles(k).name;
    fullFileName = fullfile(theFiles(k).folder, baseFileName);
    
    tf = contains(fullFileName,'MCR_1');
    T = readtable(fullFileName);
    if tf ==0
        %fprintf(1, 'Now reading %s\n', fullFileName);
        T = readtable(fullFileName);
        %C1 is teh first column for permanent gasses, H2 N2 CO O2 CR2
        %C2 is for CH4 and CO2 CR3
        %C3 is for C2 gasses like ethane CR4
        %C4 is for C3 gasses CR6
        C1 = contains(fullFileName,'MCR_2');
        C2 = contains(fullFileName,'MCR_3');
        C3 = contains(fullFileName,'MCR_4');
        C4 = contains(fullFileName,'MCR_6');
        if C1 == 1
            for l=1:height(T)
                compound = string([T.(2)(l)]);
                if compound == 'N2'
                    N2 = [N2; T.(7)(l)];
                end
                if compound == 'H2'
                    H2 = [H2; T.(7)(l)];
                end
                if compound == 'CO'
                    CO = [CO; T.(7)(l)];                   
                end
                if compound == 'O2'
                    O2 = [O2; T.(7)(l)];
                end
            end
        end
        if C2 == 1
            for l=1:8
                compound = string([T.(2)(l)]);
                if compound == 'CO2'
                   CO2 = [CO2; T.(7)(l)];
                end
                if compound == 'CH4'
                    CH4 = [CH4; T.(7)(l)];
                end
            end          
        end
        if C3 == 1 
             for l=1:height(T)
                compound = string([T.(2)(l)]);
                if compound == 'C2H6'
                   C2H6 = [C2H6; T.(7)(l)];
                end
                if compound == 'C2H4'
                    C2H4 = [C2H4; T.(7)(l)];
                end
                if compound == 'C2H2'
                    C2H2 = [C2H2; T.(7)(l)];
                end
             end            
        end
        if C4 == 1
             for l=1:height(T)
                compound = string([T.(2)(l)]);
                if compound == 'C3H8'
                   C3H8 = [C3H8; T.(7)(l)];
                end
                if compound == 'CH3OCH3'
                    
                    CH3OCH3 = [CH3OCH3; T.(7)(l)];
                end
                if compound == 'CH3OH'
                    CH3OH = [CH3OH; T.(7)(l)];
                end
                if compound == 'CH3CH2OH'
                    CH3CH2OH = [CH3CH2OH; T.(7)(l)];
                end
             end
        end
    end
end

%% samenvoegen allen concentraties
All_the_concentrations = table(N2,CH4,CO2,CO,H2,O2,C2H6,C2H4,C2H2,C3H8,CH3OCH3,CH3OH,CH3CH2OH);
Vars = ["N2";"CH4"; "CO2"; "CO"; "H2";"O2"; "C2H6"; "C2H4"; "C2H2"; "C3H8"; "CH3OCH3";"CH3OH";"CH3CH2OH"];

%% uitgooien data outlieers
% replaces the outliers with 0
B = filloutliers(All_the_concentrations,0,"quartiles");

%% printen die handel
Y = circshift(B,9); % this i needed because the order was 10,11,1,2,3,4,5,6,7,8,9
writetable(Y,outputfile,'Sheet',1,'Range','A1');
disp(Y)
disp('done')

In [ ]:
import os
import pandas as pd
import numpy as np

# Define the folder where the files live
my_folder = r'C:\Users\woute\Desktop\MA-20007-500\GC\recal\2023-03-16-MA-20007-500-01-03-Plasma.rslt'
outputfile = r'C:\Users\woute\Desktop\MA-20007-500\GC\recal\Plasma-03_outlier.xlsx'

# Make sure that folder actually exists
if not os.path.isdir(my_folder):
    print(f"Error: The following folder does not exist:\n{my_folder}\nPlease specify a new folder.")
    my_folder = input("Enter new folder path: ")

# Initialize empty lists for each compound
N2, CH4, CO2, CO, H2, O2, C2H6, C2H4, C2H2, C3H8, CH3OCH3, CH3OH, CH3CH2OH = [], [], [], [], [], [], [], [], [], [], [], [], []

# Get a list of all .csv files in the folder
file_list = [f for f in os.listdir(my_folder) if f.endswith('.csv')]

for filename in file_list:
    file_path = os.path.join(my_folder, filename)
    if "MCR_1" not in file_path:
        df = pd.read_csv(file_path)
        # Different conditions for different files
        for condition, compounds, list_comp in zip(["MCR_2", "MCR_3", "MCR_4", "MCR_6"],
                                                   [["N2", "H2", "CO", "O2"], ["CO2", "CH4"], ["C2H6", "C2H4", "C2H2"], ["C3H8", "CH3OCH3", "CH3OH", "CH3CH2OH"]],
                                                   [[N2, H2, CO, O2], [CO2, CH4], [C2H6, C2H4, C2H2], [C3H8, CH3OCH3, CH3OH, CH3CH2OH]]):
            if condition in file_path:
                for compound, list_c in zip(compounds, list_comp):
                    list_c.extend(df.loc[df.iloc[:, 1] == compound, df.columns[6]].values.tolist())

# Combine all concentrations into a dataframe
all_concentrations = pd.DataFrame(data={"N2": N2, "CH4": CH4, "CO2": CO2, "CO": CO, "H2": H2, "O2": O2, "C2H6": C2H6,
                                        "C2H4": C2H4, "C2H2": C2H2, "C3H8": C3H8, "CH3OCH3": CH3OCH3, "CH3OH": CH3OH, "CH3CH2OH": CH3CH2OH})

# Replace outliers with 0
B = all_concentrations.apply(lambda s: pd.Series(np.where(np.abs(s - s.median()) > 1.5 * (s.quantile(.75) - s.quantile(.25)), 0, s)))

# Shift columns and write to output
Y = B[B.columns.to_list()[9:] + B.columns.to_list()[:9]]
Y.to_excel(outputfile, index=False)
print(Y)
print('done')